# Hindi voice-clone detection: train a detector and check it for shortcuts

**How to run:** Runtime → Change runtime type → T4 GPU, then Runtime → Run all. Allow Google Drive access when asked. The whole notebook takes about 25 minutes the first time and much less after that, because the slow part is saved to Drive.

Needs the dataset backup `hindi16k.zip` in `MyDrive/voice-clone-detector`, made by `01_dataset_and_baseline.ipynb`.

## 1. Session setup

Restores the dataset from Drive, loads AASIST (the English-trained baseline) and defines the scoring code.

In [ ]:
import os, json, shutil, subprocess, importlib.util
import numpy as np, pandas as pd, soundfile as sf, torch
from sklearn.metrics import roc_curve
from tqdm.auto import tqdm

DRIVE = "/content/drive/MyDrive/voice-clone-detector"
OUT16 = "/content/data/hindi16k"
AASIST_DIR = "/content/aasist"
RESULTS = f"{DRIVE}/results"

# --- 1. Google Drive ---------------------------------------------------------
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.makedirs(RESULTS, exist_ok=True)

# --- 2. Dataset: restore from the Drive backup if this session does not have it
if not os.path.exists(f"{OUT16}/meta.csv"):
    backup = f"{DRIVE}/hindi16k.zip"
    assert os.path.exists(backup), "hindi16k.zip is not in your Drive folder. Stop here and tell Claude."
    print("restoring the dataset from Google Drive ...")
    shutil.copy(backup, "/content/hindi16k.zip")
    shutil.unpack_archive("/content/hindi16k.zip", OUT16)
meta = pd.read_csv(f"{OUT16}/meta.csv")
missing = [p for p in meta.path if not os.path.exists(p)]
assert not missing, f"{len(missing)} clips are missing, for example {missing[0]}"

# --- 3. AASIST, the baseline detector ---------------------------------------
if not os.path.exists(AASIST_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/clovaai/aasist", AASIST_DIR], check=True)
spec = importlib.util.spec_from_file_location("aasist_model", f"{AASIST_DIR}/models/AASIST.py")
aasist = importlib.util.module_from_spec(spec)
spec.loader.exec_module(aasist)

device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = json.load(open(f"{AASIST_DIR}/config/AASIST.conf"))["model_config"]
model = aasist.Model(cfg).to(device)
model.load_state_dict(torch.load(f"{AASIST_DIR}/models/weights/AASIST.pth", map_location=device))
model.eval()

# --- 4. Shared loading and scoring code -------------------------------------
CLIP_LEN = 64600      # about 4 seconds, the length AASIST was trained on
TARGET_DB = -23.0     # every clip is set to this loudness


def load_clip(path):
    a, sr = sf.read(path, dtype="float32")
    assert sr == 16000, f"{path} is {sr} Hz"
    a = a - a.mean()
    a = a * (10 ** (TARGET_DB / 20) / (np.sqrt(np.mean(a ** 2)) + 1e-9))
    if len(a) < CLIP_LEN:
        a = np.tile(a, CLIP_LEN // len(a) + 1)
    return a[:CLIP_LEN]


def score_clips(paths, batch=32, desc="scoring"):
    """Returns one number per clip. Higher means the detector thinks it is real."""
    realness, says_real = [], []
    for i in tqdm(range(0, len(paths), batch), desc=desc):
        x = torch.from_numpy(np.stack([load_clip(p) for p in paths[i:i + batch]])).to(device)
        with torch.no_grad():
            _, out = model(x)
        realness += out[:, 1].cpu().tolist()
        says_real += (out[:, 1] > out[:, 0]).cpu().tolist()
    return np.array(realness), np.array(says_real)


def eer(is_real, score):
    """Equal Error Rate in percent. 0 is perfect, 50 is a coin flip."""
    fpr, tpr, _ = roc_curve(is_real, score)
    i = np.nanargmin(np.abs((1 - tpr) - fpr))
    return 100 * (fpr[i] + 1 - tpr[i]) / 2


def eer_range(is_real, score, n=500, seed=0):
    """Rough 95% range for the EER, from re-drawing the test clips many times."""
    rng, vals = np.random.default_rng(seed), []
    for _ in range(n):
        j = rng.integers(0, len(score), len(score))
        if 0 < is_real[j].sum() < len(j):
            vals.append(eer(is_real[j], score[j]))
    return np.percentile(vals, [2.5, 97.5])


print("\n=== SETUP ===")
print("Running on:", "GPU" if device == "cuda" else "CPU (slow: switch the runtime to T4 GPU and run this again)")
print("Clips:", len(meta), "| test clips:", (meta.split == "test").sum())
print("Results folder:", RESULTS)
print("Files already there:", sorted(os.listdir(RESULTS)))

## 2. Compression test

Makes voice-note and phone-call versions of every test clip and scores them with AASIST.

In [ ]:
import os, subprocess, tempfile
from concurrent.futures import ThreadPoolExecutor

COND_DIR = "/content/data/hindi16k_conditions"
PHONE_BAND = "highpass=f=300,lowpass=f=3400"      # the narrow band a phone line carries

# name: (how ffmpeg encodes it, file type)
CONDITIONS = {
    "opus_24k":   (["-c:a", "libopus", "-b:a", "24k", "-application", "voip"], "ogg"),   # voice note
    "opus_12k":   (["-c:a", "libopus", "-b:a", "12k", "-application", "voip"], "ogg"),   # voice note, poor connection
    "phone_g711": (["-af", PHONE_BAND, "-ar", "8000", "-c:a", "pcm_alaw"], "wav"),       # landline / internet call
    "phone_gsm":  (["-af", PHONE_BAND, "-ar", "8000", "-c:a", "libgsm"], "gsm"),         # 2G mobile call
}

have = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True).stdout
for name in list(CONDITIONS):
    codec = CONDITIONS[name][0][CONDITIONS[name][0].index("-c:a") + 1]
    if f" {codec} " not in have:
        print(f"skipping {name}: this ffmpeg has no {codec} encoder")
        del CONDITIONS[name]


def degrade(src, dst, args, ext):
    """Encode a clip the way a call or voice note would, then decode it back to 16 kHz WAV."""
    if os.path.exists(dst):
        return
    with tempfile.TemporaryDirectory() as tmp:
        mid = f"{tmp}/x.{ext}"
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", src, "-ac", "1", *args, mid], check=True)
        subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", mid, "-ac", "1", "-ar", "16000",
                        "-c:a", "pcm_s16le", dst + ".tmp.wav"], check=True)
    os.replace(dst + ".tmp.wav", dst)


test = meta[meta.split == "test"].reset_index(drop=True)
test["is_real"] = (test.label == "real").astype(int)

# --- 1. Make a degraded copy of every test clip, for every condition ---------
paths = {"clean": list(test.path)}
for name, (args, ext) in CONDITIONS.items():
    for label in ("fake", "real"):
        os.makedirs(f"{COND_DIR}/{name}/{label}", exist_ok=True)
    paths[name] = [f"{COND_DIR}/{name}/{label}/{os.path.basename(p)}" for p, label in zip(test.path, test.label)]
    with ThreadPoolExecutor(max_workers=os.cpu_count() or 2) as pool:
        jobs = [pool.submit(degrade, s, d, args, ext) for s, d in zip(test.path, paths[name])]
        for j in tqdm(jobs, desc=f"encoding {name}"):
            j.result()

# --- 2. Score every condition with AASIST ------------------------------------
rows, scores = [], []
for name, plist in paths.items():
    realness, says_real = score_clips(plist, desc=f"scoring {name}")
    d = test[["path", "label", "tool", "speaker_id", "is_real"]].assign(
        condition=name, cond_path=plist, realness=realness, says_real=says_real)
    scores.append(d)
    real = d[d.label == "real"]
    row = {"condition": name, "EER all": eer(d.is_real.to_numpy(), d.realness.to_numpy())}
    for t in ("freevc24", "xtts_v2"):
        sub = pd.concat([real, d[d.tool == t]])
        row[f"EER {t}"] = eer(sub.is_real.to_numpy(), sub.realness.to_numpy())
    row["real called fake %"] = 100 * (1 - real.says_real.mean())
    for t in ("freevc24", "xtts_v2"):
        row[f"{t} passed as real %"] = 100 * d[d.tool == t].says_real.mean()
    rows.append(row)

summary = pd.DataFrame(rows).set_index("condition").round(1)
pd.concat(scores).to_csv(f"{RESULTS}/aasist_conditions_hindi_scores.csv", index=False)
summary.to_csv(f"{RESULTS}/aasist_conditions_hindi_summary.csv")

print("\n=== AASIST on Hindi test clips, clean and degraded ===")
print(summary.to_string())
print(f"\n{len(test)} test clips per condition. Saved to {RESULTS}")

## 3. Train a detector on clean Hindi

A pretrained multilingual speech model (XLS-R, 128 languages) turns each clip into features; a small classifier learns real against fake from the 4,158 training clips. Tested on clean and degraded audio, and with one cloning tool left out of training.

In [ ]:
import copy
import torch.nn as nn
from transformers import Wav2Vec2Model

assert "paths" in globals() and "clean" in paths, "Run the Step 8 cell first: it makes the degraded test clips."

SSL_NAME = "facebook/wav2vec2-xls-r-300m"     # speech model pretrained on 128 languages
FEAT_DIR = f"{DRIVE}/features"      # saved to Drive so a reset does not cost the slow part
MODELS = f"{DRIVE}/models"
os.makedirs(FEAT_DIR, exist_ok=True)
os.makedirs(MODELS, exist_ok=True)

# --- 1. Turn every clip into features with the pretrained speech model -------
ssl_model = Wav2Vec2Model.from_pretrained(SSL_NAME).to(device).eval()


def embed(path_list, name):
    """For each clip: one vector per layer of the speech model, averaged over time."""
    cache = f"{FEAT_DIR}/{name}.npy"
    if os.path.exists(cache):
        return np.load(cache)
    out = []
    for i in tqdm(range(0, len(path_list), 32), desc=f"features {name}"):
        x = np.stack([load_clip(p) for p in path_list[i:i + 32]])
        x = (x - x.mean(1, keepdims=True)) / (x.std(1, keepdims=True) + 1e-7)
        with torch.no_grad():
            layers = ssl_model(torch.from_numpy(x).to(device), output_hidden_states=True).hidden_states
        out.append(torch.stack([h.mean(1) for h in layers], 1).cpu().numpy().astype("float16"))
    out = np.concatenate(out)
    assert np.isfinite(out).all(), "features contain NaN or infinity"
    np.save(cache, out)
    return out


F_all = embed(list(meta.path), "clean_all")
is_real_all = (meta.label == "real").to_numpy().astype(int)
tool_all, split_all = meta.tool.to_numpy(), meta.split.to_numpy()
test_mask = split_all == "test"
assert list(meta.path[test_mask]) == paths["clean"], "test clips are not in the expected order"
F_test = {"clean": F_all[test_mask]}
for name in paths:
    if name != "clean":
        F_test[name] = embed(paths[name], f"test_{name}")
y_test, tool_test = is_real_all[test_mask], tool_all[test_mask]


# --- 2. A small classifier on top of those features --------------------------
class Head(nn.Module):
    """Learns how much to trust each layer, then draws one line between real and fake."""

    def __init__(self, n_layers, dim):
        super().__init__()
        self.layer_weights = nn.Parameter(torch.zeros(n_layers))
        self.drop = nn.Dropout(0.2)
        self.out = nn.Linear(dim, 1)

    def forward(self, f):
        w = torch.softmax(self.layer_weights, 0)[None, :, None]
        return self.out(self.drop((w * f).sum(1))).squeeze(-1)


def eer_threshold(is_real, score):
    fpr, tpr, thr = roc_curve(is_real, score)
    return thr[np.nanargmin(np.abs((1 - tpr) - fpr))]


def train_head(tools, epochs=60, seed=0):
    """Train on real clips plus fakes from the given tools. Returns a scoring function and details."""
    torch.manual_seed(seed)
    use = np.isin(tool_all, ["real", *tools])
    tr, va = use & (split_all == "train"), use & (split_all == "val")
    mu = torch.tensor(F_all[tr].astype("float32").mean(0, keepdims=True), device=device)
    sd = torch.tensor(F_all[tr].astype("float32").std(0, keepdims=True) + 1e-5, device=device)
    prep = lambda F: (torch.tensor(F.astype("float32"), device=device) - mu) / sd
    Xtr, ytr = prep(F_all[tr]), torch.tensor(is_real_all[tr], dtype=torch.float32, device=device)
    Xva, yva = prep(F_all[va]), is_real_all[va]

    head = Head(F_all.shape[1], F_all.shape[2]).to(device)
    opt = torch.optim.Adam(head.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=(1 - ytr).sum() / ytr.sum())
    best = (101.0, None, 0)
    for epoch in range(1, epochs + 1):
        head.train()
        order = torch.randperm(len(Xtr), device=device)
        for i in range(0, len(order), 64):
            j = order[i:i + 64]
            opt.zero_grad()
            loss_fn(head(Xtr[j]), ytr[j]).backward()
            opt.step()
        head.eval()
        with torch.no_grad():
            val_eer = eer(yva, head(Xva).cpu().numpy())
        if val_eer < best[0]:
            best = (val_eer, copy.deepcopy(head.state_dict()), epoch)
    head.load_state_dict(best[1])
    head.eval()

    def score(F):
        with torch.no_grad():
            return head(prep(F)).cpu().numpy()

    return dict(score=score, head=head, mu=mu, sd=sd, val_eer=best[0], epoch=best[2],
                threshold=eer_threshold(yva, score(F_all[va])), n_train=int(tr.sum()))


def eer_vs_real(scores, tool):
    keep = (tool_test == "real") | (tool_test == tool)
    return eer(y_test[keep], scores[keep])


# --- 3. Train on clean Hindi (both tools), test on clean and degraded audio --
both = train_head(["freevc24", "xtts_v2"])
rows, out_scores = [], []
for name, F in F_test.items():
    s = both["score"](F)
    says_real = s >= both["threshold"]
    row = {"condition": name, "EER all": eer(y_test, s), "EER freevc24": eer_vs_real(s, "freevc24"),
           "EER xtts_v2": eer_vs_real(s, "xtts_v2"),
           "real called fake %": 100 * (1 - says_real[y_test == 1].mean()),
           "fake passed as real %": 100 * says_real[y_test == 0].mean()}
    if "summary" in globals() and name in summary.index:
        row["AASIST EER all"] = summary.loc[name, "EER all"]
    rows.append(row)
    out_scores.append(pd.DataFrame(dict(path=paths[name], condition=name, tool=tool_test, is_real=y_test,
                                        realness=s, says_real=says_real)))
table_a = pd.DataFrame(rows).set_index("condition").round(1)

# --- 4. Same model, but trained on one cloning tool and tested on the other --
rows = [{"trained on": "both tools", "EER freevc24": eer_vs_real(both["score"](F_test["clean"]), "freevc24"),
         "EER xtts_v2": eer_vs_real(both["score"](F_test["clean"]), "xtts_v2"), "unseen tool": "none"}]
for seen, unseen in (("freevc24", "xtts_v2"), ("xtts_v2", "freevc24")):
    one = train_head([seen])
    s = one["score"](F_test["clean"])
    rows.append({"trained on": f"{seen} only", "EER freevc24": eer_vs_real(s, "freevc24"),
                 "EER xtts_v2": eer_vs_real(s, "xtts_v2"), "unseen tool": unseen})
table_b = pd.DataFrame(rows).set_index("trained on").round(1)

# --- 5. Save and report -------------------------------------------------------
pd.concat(out_scores).to_csv(f"{RESULTS}/xlsr_clean_trained_scores.csv", index=False)
table_a.to_csv(f"{RESULTS}/xlsr_clean_trained_by_condition.csv")
table_b.to_csv(f"{RESULTS}/xlsr_clean_trained_unseen_tool.csv")
torch.save({"head": both["head"].state_dict(), "mu": both["mu"].cpu(), "sd": both["sd"].cpu(),
            "threshold": float(both["threshold"]), "ssl_name": SSL_NAME}, f"{MODELS}/xlsr_head_clean.pt")

lw = torch.softmax(both["head"].layer_weights, 0).detach().cpu().numpy()
print("\n=== YOUR DETECTOR, trained on clean Hindi only ===")
print(f"Trained on {both['n_train']} clips. Validation EER {both['val_eer']:.1f}% (best at epoch {both['epoch']}).")
print("Layers it relies on most:", ", ".join(f"{i} ({100 * lw[i]:.0f}%)" for i in np.argsort(-lw)[:3]),
      f"of {len(lw)}")
print("\nA. Tested on clean and degraded audio (decisions use one fixed threshold set on clean validation clips):")
print(table_a.to_string())
print("\nB. Clean audio, when one cloning tool was never seen in training:")
print(table_b.to_string())
print(f"\nSaved to {RESULTS} and {MODELS}")

## 4. Is it a shortcut?

Every real clip comes from one source (Kathbath phone recordings) and every fake from another (IndicSynth), so a detector could learn where a recording came from instead of whether it is cloned. Two checks:

1. Seven basic audio measurements with no AI: if they also separate real from fake, the dataset has an easy giveaway.
2. Real Hindi speech from two sources the detector never saw (FLEURS and IndicTTS studio recordings): if it calls these fake, it learned the recording source.

In [ ]:
import io, os, glob
from math import gcd
import pyarrow.parquet as pq
from scipy.signal import resample_poly
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from huggingface_hub import hf_hub_download

assert "both" in globals(), "Run the Step 9 cell first: this one checks the detector it trained."
OOD = "/content/data/other_real"

# --- 1. Can a few basic audio measurements, with no AI, tell real from fake? --
def basic_measurements(path):
    """Seven plain numbers about a clip: background level, dynamic range and spectral shape."""
    a = load_clip(path)
    frames = np.lib.stride_tricks.sliding_window_view(a, 512)[::256] * np.hanning(512)
    spec = np.abs(np.fft.rfft(frames, axis=1)) ** 2 + 1e-12
    freqs = np.fft.rfftfreq(512, 1 / 16000)
    frame_db = 10 * np.log10(spec.sum(1))
    total = spec.sum(1)
    cum = np.cumsum(spec, 1) / total[:, None]
    return [np.percentile(frame_db, 10),                                    # background level
            np.percentile(frame_db, 90) - np.percentile(frame_db, 10),      # dynamic range
            np.mean((spec * freqs).sum(1) / total),                         # brightness
            np.mean(freqs[np.argmax(cum >= 0.85, axis=1)]),                 # where 85% of energy sits
            np.mean(np.exp(np.log(spec).mean(1)) / spec.mean(1)),           # noisiness
            np.mean(spec[:, freqs > 4000].sum(1) / total),                  # share above 4 kHz
            np.mean(np.abs(np.diff(np.sign(a))) > 0)]                       # zero crossings


B_all = np.array([basic_measurements(p) for p in tqdm(list(meta.path), desc="measuring clips")])
simple = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
simple.fit(B_all[split_all == "train"], is_real_all[split_all == "train"])
simple_rows = []
for name, plist in paths.items():
    B = B_all[test_mask] if name == "clean" else np.array([basic_measurements(p) for p in plist])
    simple_rows.append({"condition": name, "7 basic measurements, EER": eer(y_test, simple.decision_function(B)),
                        "your detector, EER": table_a.loc[name, "EER all"]})
simple_table = pd.DataFrame(simple_rows).set_index("condition").round(1)


# --- 2. Real Hindi speech from other recordings: does the detector call it fake?
def take_real_clips(repo, files, revision, n_per_file, source):
    """Download parquet files, keep n evenly spaced clips from each, save them as 16 kHz WAV."""
    folder = f"{OOD}/{source}"
    done = sorted(glob.glob(f"{folder}/*.wav")) if os.path.isdir(folder) else []
    if done:
        return done
    os.makedirs(folder, exist_ok=True)
    out = []
    for fname in files:
        print("downloading", repo, fname, "...")
        local = hf_hub_download(repo, fname, repo_type="dataset", revision=revision, local_dir="/content/hf_tmp")
        n = pq.ParquetFile(local).metadata.num_rows
        want = set(np.unique(np.linspace(0, n - 1, min(n_per_file, n)).astype(int)).tolist())
        start = 0
        for batch in pq.ParquetFile(local).iter_batches(batch_size=100, columns=["audio"]):
            for i, rec in enumerate(batch.column("audio").to_pylist()):
                if start + i in want and rec and rec.get("bytes"):
                    a, sr = sf.read(io.BytesIO(rec["bytes"]), dtype="float32", always_2d=True)
                    a = a.mean(axis=1)
                    if sr != 16000:
                        g = gcd(sr, 16000)
                        a = resample_poly(a, 16000 // g, sr // g)
                    if len(a) >= 16000:                                       # skip clips under 1 second
                        p = f"{folder}/{source}_{len(out):04d}.wav"
                        sf.write(p, np.clip(a, -1, 1).astype("float32"), 16000, subtype="PCM_16")
                        out.append(p)
            start += batch.num_rows
        os.remove(local)
    return out


other = {
    "FLEURS (read speech, own devices)": take_real_clips(
        "google/fleurs", ["hi_in/test/0000.parquet"], "refs/convert/parquet", 300, "fleurs"),
    "IndicTTS (studio recordings)": take_real_clips(
        "SPRINGLab/IndicTTS-Hindi", ["data/train-00002-of-00010.parquet", "data/train-00006-of-00010.parquet"],
        None, 150, "indictts"),
}

clean_scores = both["score"](F_test["clean"])
fake_scores = clean_scores[y_test == 0]
ood_rows = [{"real speech from": "Kathbath test set (same source as training)", "clips": int((y_test == 1).sum()),
             "called fake %": 100 * (clean_scores[y_test == 1] < both["threshold"]).mean(),
             "EER vs the test fakes": eer(y_test, clean_scores)}]
for label, plist in other.items():
    s = both["score"](embed(plist, "other_" + label.split()[0].lower()))
    ood_rows.append({"real speech from": label, "clips": len(plist),
                     "called fake %": 100 * (s < both["threshold"]).mean(),
                     "EER vs the test fakes": eer(np.r_[np.ones(len(s)), np.zeros(len(fake_scores))],
                                                  np.r_[s, fake_scores])})
ood_table = pd.DataFrame(ood_rows).set_index("real speech from").round(1)

simple_table.to_csv(f"{RESULTS}/shortcut_basic_measurements.csv")
ood_table.to_csv(f"{RESULTS}/shortcut_other_real_speech.csv")

print("\n=== IS IT A SHORTCUT? ===")
print("\n1. Seven basic audio measurements (background level, loudness range, spectral shape), no AI:")
print(simple_table.to_string())
print("\n2. Real Hindi speech from recordings the detector never saw, scored with the Step 9 threshold:")
print(ood_table.to_string())
print(f"\nSaved to {RESULTS}")